# DZ4 — Масштабування даних в економічних дослідженнях


In [ ]:
import pandas as pd

data = {
    'year': list(range(2010, 2021)),
    'index': [82, 88, 94, 101, 108, 116, 123, 131, 138, 144, 148]
}
df = pd.DataFrame(data)
df

**Дослідження:**  
Ми створюємо **часовий ряд** з двох стовпчиків:  
- `year` — послідовні календарні роки (від 2010 до 2020 включно).  
- `index` — **індекс споживчих цін** за кожен рік.  
Індекс тут — це **накопичений ефект зростання цін** від базового року, тобто, що більший `index`, то **вища сукупна ціна** фіксованого набору товарів та послуг **у порівнянні з базовим роком**.  
Таблиця `df` дозволяє нам далі обчислити похідні показники (масштабування та річну інфляцію) і візуально перевіряти зміни за роками.

In [ ]:
df['minmaxscaler'] = df['index'].transform(
    lambda item: (item - item.min()) / (item.max() - item.min())
)
df

**Дослідження:**  
Мета — привести `index` до шкали **0–1**, щоб порівнювати **відносні** рівні незалежно від одиниць.  
Використовуємо вираз: `(item - item.min()) / (item.max() - item.min())`, де `item` — це стовпчик `index`.  
- `item.min()` — найменше значення індексу за період → стає `0`.  
- `item.max()` — найбільше значення індексу за період → стає `1`.  
Усі інші роки займають позиції **між 0 і 1**, показуючи *наскільки високо* їхній індекс відносно мінімуму та максимуму періоду.  
Робимо це через `transform(lambda ...)`.

In [ ]:
df['yearinf'] = df['index'].transform(
    lambda item: (item - item.shift(1)) / item.shift(1) * 100
)
df

**Дослідження:**  
Додаємо стовпець **`yearinf`** — це **річна інфляція у %**, тобто **швидкість, з якою зростає індекс** порівняно з попереднім роком.  
За формулою:  yearinf = (поточний індекс − попередній індекс) / попередній індекс × 100.  
У коді це реалізовано через `transform` і `shift(1)`:  
- `item.shift(1)` зсуває значення `index` на 1 рядок донизу, щоб у поточному рядку взяти **попередній індекс** (значення попереднього року).  
- Різниця `item - item.shift(1)` — це приріст індексу за рік.  
- Ділення на `item.shift(1)` і множення на `100` дає **відсоткову зміну**.  
Для **першого року** попереднього значення немає, тому з'являється `NaN` — це **очікувана** поведінка.

In [ ]:
df_rounded = df.round(2)
df_rounded

**Дослідження:**  
`round(2)` повертає копію таблиці з округленням **до 2 знаків** після коми для всіх числових стовпчиків.  
Це як окреме відображення результатів.

In [ ]:
import plotly.graph_objects as go

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=df['year'], y=df['index'], mode='lines+markers', name='index'))
fig1.add_trace(go.Scatter(x=df['year'], y=df['minmaxscaler'], mode='lines+markers', name='minmaxscaler'))
fig1.update_layout(title='Динаміка індексу та нормалізованих значень (0–1)', xaxis_title='year', yaxis_title='value')
fig1

**Дослідження:**   
Порівнюємо дві серії(стовбчики) **index** та **minmaxscaler**
- Використовуємо `import plotly.graph_objects as go`, `fig1 = go.Figure()`.  
- Додаємо дві лінії з маркерами: **index** та **minmaxscaler** (`go.Scatter(...)`).  
- Підписи осей і заголовок задаємо через `fig1.update_layout(...)`.  
Отримуємо **інтерактивний** графік де можна наводити курсор і бачити точні значення.

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.boxplot(df['yearinf'][1:], vert=False)
plt.ylabel('yearinf (%)')
plt.title('Розподіл річної інфляції (yearinf)')
plt.show()

**Дослідження:**  
Використовуємо `plt.boxplot(...)` для відображення **розподілу** значень `yearinf` (річна інфляція, %) за весь період.  
- Середня вертикальна лінія в коробці є **медіаною**.  
- Межі коробки — **1-й** та **3-й** квартилі (Q1 і Q3).  
- "Вуса" показують розкид даних поза квартилями.  
Вони тягнуться до найближчих значень, що вкладаються у діапазон Q1−1.5хIQR, Q3+1.5хIQR, де IQR = Q3 − Q1.  
Точки поза цим діапазоном показуються як окремі маркери (потенційні викиди).  
- *Примітка:* Перший рік має відсутнє значення (немає попереднього індексу), тому для boxplot беремо зріз `yearinf` **з другого елемента** — `df['yearinf'][1:]`.

In [ ]:
import plotly.graph_objects as go

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df['year'], y=df['yearinf'], mode='lines+markers', name='yearinf'))
fig2.update_layout(title='Річна інфляція (yearinf), %', xaxis_title='year', yaxis_title='yearinf (%)')
fig2

**Дослідження:**  
Показуємо динаміку зміни інфляції за роками: кожна точка — конкретний рік.  
Використовуємо `plotly.graph_objects` , `import ... as go`, `fig2 = go.Figure()`.  
Додаємо лінію з маркерами (`go.Scatter(...)`) для `yearinf` за роками.  
Оновлюємо підписи осей і заголовок через `fig.update_layout(...)`.  
Це дає **інтерактивний** графік де можна наводити курсор і бачити точні значення.